Step 1: Load the dataset and set up

In [12]:
library(dplyr)

url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

col_names <- c("age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
               "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target")

heart_data <- read.csv(url, header = FALSE, col.names = col_names, na.strings = "?")

head(heart_data)
str(heart_data)
summary(heart_data$trestbps)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0
2,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2
3,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1
4,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0
5,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0
6,56,1,2,120,236,0,0,178,0,0.8,1,0,3,0


'data.frame':	303 obs. of  14 variables:
 $ age     : num  63 67 67 37 41 56 62 57 63 53 ...
 $ sex     : num  1 1 1 1 0 1 0 0 1 1 ...
 $ cp      : num  1 4 4 3 2 2 4 4 4 4 ...
 $ trestbps: num  145 160 120 130 130 120 140 120 130 140 ...
 $ chol    : num  233 286 229 250 204 236 268 354 254 203 ...
 $ fbs     : num  1 0 0 0 0 0 0 0 0 1 ...
 $ restecg : num  2 2 2 0 2 0 2 0 2 2 ...
 $ thalach : num  150 108 129 187 172 178 160 163 147 155 ...
 $ exang   : num  0 1 1 0 0 0 0 1 0 1 ...
 $ oldpeak : num  2.3 1.5 2.6 3.5 1.4 0.8 3.6 0.6 1.4 3.1 ...
 $ slope   : num  3 2 2 3 1 1 3 1 2 3 ...
 $ ca      : num  0 3 2 0 0 0 2 0 1 0 ...
 $ thal    : num  6 3 7 3 3 3 3 3 7 7 ...
 $ target  : int  0 2 1 0 0 0 3 0 2 1 ...


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   94.0   120.0   130.0   131.7   140.0   200.0 

Step 2: Introduce simulated data problems

In [13]:
set.seed(123)

heart_data$trestbps[c(5, 15, 25)] <- -heart_data$trestbps[c(5, 15, 25)]

heart_data$trestbps[c(35, 45, 55, 65)] <- NA

heart_data$trestbps[c(75, 85)] <- c(310, 330)

summary(heart_data$trestbps)
sum(is.na(heart_data$trestbps))

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
 -172.0   120.0   130.0   130.2   140.0   330.0       4 

[1] 4

Step 3: BP-cleaning function using if-else

In [14]:
clean_bp <- function(bp_vector) {
  cleaned <- bp_vector
  for (i in seq_along(cleaned)) {
    if (is.na(cleaned[i])) {
      next
    } else if (cleaned[i] < 0) {
      cleaned[i] <- NA
    } else if (cleaned[i] > 250) {
      cleaned[i] <- 250
    }
  }
  return(cleaned)
}

heart_data$trestbps_cleaned <- clean_bp(heart_data$trestbps)

summary(heart_data$trestbps_cleaned)
sum(is.na(heart_data$trestbps_cleaned))

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
   94.0   120.0   130.0   132.5   140.0   250.0       7 

[1] 7

Step 4: tryCatch() error handling

In [15]:
safe_mean_bp <- function(bp_vector) {
  result <- tryCatch({
    if (all(is.na(bp_vector))) {
      stop("All values are NA, cannot compute mean")
    }
    mean(bp_vector, na.rm = TRUE)
  },
  warning = function(w) {
    message("Warning occurred: ", conditionMessage(w))
    NA
  },
  error = function(e) {
    message("Error occurred: ", conditionMessage(e))
    NA
  })
  return(result)
}

safe_mean_bp(heart_data$trestbps_cleaned)

[1] 132.5338

In [16]:
safe_ratio <- function(numerator, denominator) {
  result <- tryCatch({
    if (is.na(numerator) || is.na(denominator)) {
      stop("Numerator or denominator is NA")
    }
    if (denominator == 0) {
      stop("Denominator is zero, cannot divide")
    }
    numerator / denominator
  },
  error = function(e) {
    message("Could not compute ratio: ", conditionMessage(e))
    NA
  })
  return(result)
}

heart_data$chol_bp_ratio <- mapply(safe_ratio, heart_data$chol, heart_data$trestbps_cleaned)

summary(heart_data$chol_bp_ratio)

Could not compute ratio: Numerator or denominator is NA

Could not compute ratio: Numerator or denominator is NA

Could not compute ratio: Numerator or denominator is NA

Could not compute ratio: Numerator or denominator is NA

Could not compute ratio: Numerator or denominator is NA

Could not compute ratio: Numerator or denominator is NA

Could not compute ratio: Numerator or denominator is NA



   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
  0.788   1.576   1.863   1.895   2.160   4.904       7 

Step 5: Loop vs. vectorized comparison

In [17]:
bp_raw <- heart_data$trestbps

loop_time <- system.time({
  invalid_loop <- c()
  for (i in seq_along(bp_raw)) {
    if (!is.na(bp_raw[i]) && (bp_raw[i] < 0 || bp_raw[i] > 250)) {
      invalid_loop <- c(invalid_loop, i)
    }
  }
})

loop_time
length(invalid_loop)

   user  system elapsed 
  0.006   0.000   0.007 

[1] 5

In [18]:
vectorized_time <- system.time({
  invalid_vectorized <- which(!is.na(bp_raw) & (bp_raw < 0 | bp_raw > 250))
})

vectorized_time
length(invalid_vectorized)

   user  system elapsed 
  0.000   0.000   0.001 

[1] 5

In [19]:
identical(invalid_loop, invalid_vectorized)

comparison <- data.frame(
  Method = c("Loop", "Vectorized"),
  Time_Elapsed = c(loop_time["elapsed"], vectorized_time["elapsed"])
)
comparison

[1] TRUE

Method,Time_Elapsed
<chr>,<dbl>
Loop,0.007
Vectorized,0.001


Step 6: Validate the cleaned data
r

In [20]:
sum(is.na(heart_data$trestbps_cleaned))

min(heart_data$trestbps_cleaned, na.rm = TRUE)
max(heart_data$trestbps_cleaned, na.rm = TRUE)
mean(heart_data$trestbps_cleaned, na.rm = TRUE)
median(heart_data$trestbps_cleaned, na.rm = TRUE)

any(heart_data$trestbps_cleaned < 0, na.rm = TRUE)
any(heart_data$trestbps_cleaned > 250, na.rm = TRUE)

[1] 7

[1] 94

[1] 250

[1] 132.5338

[1] 130

[1] FALSE

[1] FALSE

Step 7: Save the cleaned CSV

In [21]:
write.csv(heart_data, "cleaned_heart_data.csv", row.names = FALSE)